# 📅 Day 02: String Matching — KMP, Rabin-Karp & Z-Algorithm

---

## The Problem: Finding a Needle in a Haystack

Given a text of length `n` and a pattern of length `m`, find where the pattern occurs.

**Brute force** checks every position — O(n*m). For a 1M character text and 1000 char pattern, that's a billion operations. We need better.

### The Key Insight Behind All Efficient Algorithms

When a mismatch happens, we've already **read some characters that matched**. Smart algorithms reuse that information instead of starting over.

Think of it like proofreading: if you're matching `abcabd` and you've matched `abcab` before failing at `d`, you already know the last `ab` matches the first `ab` of the pattern — so skip ahead!

```
╔═══════════════════════════════════════════════════════════════════════╗
║  STRING MATCHING ALGORITHMS                                        ║
║                                                                     ║
║  Brute Force    O(n*m)     Simple but slow                         ║
║  KMP            O(n+m)     Prefix function, never re-scan text     ║
║  Rabin-Karp     O(n+m)*    Rolling hash, great for multi-pattern   ║
║  Z-Algorithm    O(n+m)     Z-array, elegant for certain problems   ║
║                                                                     ║
║  * Rabin-Karp is O(n*m) worst case due to hash collisions          ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 🔧 KMP Algorithm (Knuth-Morris-Pratt)

KMP's magic is the **failure function** (also called `lps` — Longest Proper Prefix which is also Suffix).

For pattern `abcabd`:
```
  Index:   0  1  2  3  4  5
  Char:    a  b  c  a  b  d
  LPS:     0  0  0  1  2  0
                     ↑  ↑
              "ab" at start = "ab" ending here
```

When mismatch at index `j`, jump to `lps[j-1]` instead of starting over. The text pointer **never goes backward**.

In [ ]:
def build_lps(pattern):
    """Build the longest proper prefix-suffix array."""
    m = len(pattern)
    lps = [0] * m
    length = 0  # Length of previous longest prefix suffix
    i = 1
    
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]  # Don't increment i!
            else:
                lps[i] = 0
                i += 1
    return lps

def kmp_search(text, pattern):
    """
    KMP: O(n+m) guaranteed.
    Returns all starting indices of pattern in text.
    """
    n, m = len(text), len(pattern)
    if m == 0:
        return []
    
    lps = build_lps(pattern)
    result = []
    j = 0  # Pattern index
    
    for i in range(n):
        while j > 0 and text[i] != pattern[j]:
            j = lps[j - 1]  # Fall back using LPS
        
        if text[i] == pattern[j]:
            j += 1
        
        if j == m:
            result.append(i - m + 1)
            j = lps[j - 1]  # Look for next match
    
    return result

print(kmp_search("ababcababcabc", "abc"))  # [2, 7, 10]
print(build_lps("abcabd"))                  # [0, 0, 0, 1, 2, 0]

---

## 🔧 Rabin-Karp (Rolling Hash)

Instead of comparing characters, compare **hashes**. If hashes match, verify the actual strings (to handle collisions).

**Rolling hash trick:** When sliding the window by one character, update the hash in O(1):
```
  hash("bcd") = hash("abc") - 'a'*base² + 'd'
```

Rabin-Karp really shines for **multiple pattern search** — compute hash for all patterns, then one scan through text.

In [ ]:
def rabin_karp(text, pattern):
    """
    Rolling hash search. Average O(n+m), worst O(n*m).
    """
    n, m = len(text), len(pattern)
    if m > n:
        return []
    
    BASE = 256
    MOD = 10**9 + 7
    
    # Compute hash of pattern and first window
    p_hash = 0
    t_hash = 0
    power = 1  # BASE^(m-1) mod MOD
    
    for i in range(m):
        p_hash = (p_hash * BASE + ord(pattern[i])) % MOD
        t_hash = (t_hash * BASE + ord(text[i])) % MOD
        if i > 0:
            power = (power * BASE) % MOD
    
    result = []
    for i in range(n - m + 1):
        if p_hash == t_hash and text[i:i+m] == pattern:  # Verify on match
            result.append(i)
        
        # Roll the hash forward
        if i < n - m:
            t_hash = ((t_hash - ord(text[i]) * power) * BASE + ord(text[i+m])) % MOD
    
    return result

print(rabin_karp("ababcababcabc", "abc"))  # [2, 7, 10]

---

## 🎯 Problem: Implement strStr() (LC #28)

Return the index of the first occurrence of `needle` in `haystack`.

In [ ]:
def strStr(haystack, needle):
    """KMP approach. O(n+m) time."""
    matches = kmp_search(haystack, needle)
    return matches[0] if matches else -1

print(strStr("sadbutsad", "sad"))    # 0
print(strStr("leetcode", "leeto"))   # -1

---

## 🎯 Problem: Repeated Substring Pattern (LC #459)

Check if string can be formed by repeating a substring.

**Clever trick:** If `s` is made of repeated pattern `p`, then `(s+s)[1:-1]` still contains `s`. Why? Because you've overlapped enough copies.

**KMP trick:** If `n % (n - lps[-1]) == 0` and `lps[-1] > 0`, it's periodic.

In [ ]:
def repeatedSubstringPattern(s):
    """Concatenation trick: O(n) time."""
    doubled = (s + s)[1:-1]  # Remove first and last char
    return s in doubled

print(repeatedSubstringPattern("abab"))    # True  ("ab" repeated)
print(repeatedSubstringPattern("aba"))     # False
print(repeatedSubstringPattern("abcabcabc"))  # True

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  STRING MATCHING DECISION GUIDE                                    ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  Single pattern search      → KMP (guaranteed O(n+m))             ║
║  Multiple pattern search    → Rabin-Karp (hash all patterns)      ║
║  "Is repeated substring?"   → Doubling trick or KMP LPS           ║
║  Prefix-suffix questions    → Build LPS array directly            ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 28 | Find the Index of First Occurrence | Easy | KMP or built-in |
| 459 | Repeated Substring Pattern | Easy | Doubling trick |
| 214 | Shortest Palindrome | Hard | KMP on s + '#' + rev(s) |
| 686 | Repeated String Match | Medium | Rabin-Karp |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 KMP: build LPS array, text pointer never backtracks           │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Rabin-Karp: rolling hash, O(1) to slide window                │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Repeated pattern?" → doubling trick is the cleanest          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 In interviews, mention KMP even if you use Python's `in`      │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Anagram & Palindrome Patterns